In [0]:
%sql CREATE TABLE IF NOT EXISTS emp1 ( emp_id INT, first_name STRING, last_name STRING, department STRING, salary INT, hire_date DATE );

In [0]:
%sql
INSERT OVERWRITE emp1 VALUES
(1, 'Nirajan', 'kc', 'IT', 60000, '2022-01-15'),
(2, 'Keshab','Rai','HR', 45000, '2021-03-10'),
(3, 'Shalani','Shah','IT', 70000, '2020-06-01'),
(4,'Nisha','Adhikari','Finance', 50000, '2019-11-12'),
(5,'Matrika','Subedi','HR', 48000, '2022-06-01'),
(6,'Arbin','Budhathoki','IT', 55000, '2023-02-18'),
(7,'Avishek','Bhandari','Finance', 65000, '2021-09-30'),
(8,'Saroj','Neupane','IT', 72000,'2020-12-25');

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
SELECT *
FROM emp1
WHERE salary > (
    SELECT AVG(salary)
    FROM emp1
);

emp_id,first_name,last_name,department,salary,hire_date
1,Nirajan,kc,IT,60000,2022-01-15
3,Shalani,Shah,IT,70000,2020-06-01
7,Avishek,Bhandari,Finance,65000,2021-09-30
8,Saroj,Neupane,IT,72000,2020-12-25


In [0]:
%sql
SELECT department,
       first_name,
       last_name,
       salary
FROM (
    SELECT *,
           RANK() OVER (
               PARTITION BY department
               ORDER BY salary DESC
           ) AS rnk
    FROM emp1
) t
WHERE rnk = 1;

department,first_name,last_name,salary
Finance,Avishek,Bhandari,65000
HR,Matrika,Subedi,48000
IT,Saroj,Neupane,72000


In [0]:
%sql
SELECT salary
FROM (
    SELECT salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM emp1
) t
WHERE rnk = 2;

salary
70000


In [0]:
%sql
SELECT department,
       first_name,
       last_name,
       salary
FROM (
    SELECT *,
           DENSE_RANK() OVER (
               PARTITION BY department
               ORDER BY salary DESC
           ) AS rnk
    FROM emp1
) t
WHERE rnk = 2;

department,first_name,last_name,salary
Finance,Nisha,Adhikari,50000
HR,Keshab,Rai,45000
IT,Shalani,Shah,70000


In [0]:
%sql
SELECT emp_id,
       first_name,
       salary,
       hire_date,
       SUM(salary) OVER (
           ORDER BY hire_date
       ) AS running_total
FROM emp1;

emp_id,first_name,salary,hire_date,running_total
4,Nisha,50000,2019-11-12,50000
3,Shalani,70000,2020-06-01,120000
8,Saroj,72000,2020-12-25,192000
2,Keshab,45000,2021-03-10,237000
7,Avishek,65000,2021-09-30,302000
1,Nirajan,60000,2022-01-15,362000
5,Matrika,48000,2022-06-01,410000
6,Arbin,55000,2023-02-18,465000


In [0]:
%sql
SELECT emp_id,
       first_name,
       department,
       salary,
       AVG(salary) OVER (
           PARTITION BY department
       ) AS dept_avg_salary,
       salary - AVG(salary) OVER (
           PARTITION BY department
       ) AS difference
FROM emp1;

emp_id,first_name,department,salary,dept_avg_salary,difference
4,Nisha,Finance,50000,57500.0,-7500.0
7,Avishek,Finance,65000,57500.0,7500.0
2,Keshab,HR,45000,46500.0,-1500.0
5,Matrika,HR,48000,46500.0,1500.0
1,Nirajan,IT,60000,64250.0,-4250.0
3,Shalani,IT,70000,64250.0,5750.0
6,Arbin,IT,55000,64250.0,-9250.0
8,Saroj,IT,72000,64250.0,7750.0


In [0]:
%sql
SELECT emp_id,
       first_name,
       department,
       salary,
       RANK() OVER (
           PARTITION BY department
           ORDER BY salary DESC
       ) AS salary_rank
FROM emp1;

emp_id,first_name,department,salary,salary_rank
7,Avishek,Finance,65000,1
4,Nisha,Finance,50000,2
5,Matrika,HR,48000,1
2,Keshab,HR,45000,2
8,Saroj,IT,72000,1
3,Shalani,IT,70000,2
1,Nirajan,IT,60000,3
6,Arbin,IT,55000,4


In [0]:
%sql
SELECT e.*
FROM emp1 e
JOIN (
    SELECT department,
           AVG(UNIX_TIMESTAMP(hire_date)) AS avg_hire
    FROM emp1
    GROUP BY department
) d
ON e.department = d.department
WHERE UNIX_TIMESTAMP(e.hire_date) < d.avg_hire;

emp_id,first_name,last_name,department,salary,hire_date
2,Keshab,Rai,HR,45000,2021-03-10
3,Shalani,Shah,IT,70000,2020-06-01
4,Nisha,Adhikari,Finance,50000,2019-11-12
8,Saroj,Neupane,IT,72000,2020-12-25


In [0]:
%sql
SELECT department,
       COUNT(*) AS total_employees,
       MIN(salary) AS min_salary,
       MAX(salary) AS max_salary,
       AVG(salary) AS avg_salary,
       SUM(salary) AS total_salary
FROM emp1
GROUP BY department;

department,total_employees,min_salary,max_salary,avg_salary,total_salary
IT,4,55000,72000,64250.0,257000
HR,2,45000,48000,46500.0,93000
Finance,2,50000,65000,57500.0,115000


In [0]:
%sql
SELECT department,
       first_name,
       last_name,
       salary
FROM (
    SELECT *,
           DENSE_RANK() OVER (
               PARTITION BY department
               ORDER BY salary DESC
           ) AS rnk
    FROM emp1
) t
WHERE rnk <= 2
ORDER BY department, salary DESC;

department,first_name,last_name,salary
Finance,Avishek,Bhandari,65000
Finance,Nisha,Adhikari,50000
HR,Matrika,Subedi,48000
HR,Keshab,Rai,45000
IT,Saroj,Neupane,72000
IT,Shalani,Shah,70000
